# Data & Packages

In [1]:
# Main Packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scipy

# Clustering 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, adjusted_rand_score, mutual_info_score

# Parallel processing 
from joblib import Parallel, delayed

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

In [4]:
# Time conversions
seconds_in_day = 60 * 60 * 24
eight_weeks_seconds = 8 * 7 * seconds_in_day

# Time aggregation
agg_hour_level = 1

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [5]:
# Keep the first 8 weeks of data
df = df.filter(pl.col("time") <= eight_weeks_seconds)

In [6]:
# Chosen features
feature_cols = [
    'log_n_events',
    'log_n_distinct_src',
    'log_n_distinct_dest',
    'failure_ratio',
    'c_bar',
    's_bar',
    ]

# Functions

In [7]:
# Build the features dataframe
def build_features(df,agg_hour_level):

    agg_seconds = agg_hour_level * 60 * 60 

    return (
        df.with_columns(
            bucket = pl.col('time') // agg_seconds,
            theta = ((pl.col('time') % seconds_in_day)/seconds_in_day) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user','bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
            log_n_events = pl.col('n_events').log(),
            log_n_distinct_src = pl.col('n_distinct_src').log(),
            log_n_distinct_dest = pl.col('n_distinct_dest').log(),
        ).collect()
        )

In [8]:
# Process features for clustering
def cluster_preprocess(features_df, X_scaled, week):

    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    in_bin = features_df['bucket'].is_between(lb,ub).to_numpy()

    features_week = features_df.filter(in_bin)
    X_scaled_week = X_scaled[in_bin]

    return features_week, X_scaled_week 

# 1 Hour

In [ ]:
# Time aggregation
agg_hour_level = 1

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [ ]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [ ]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [ ]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 615624 week 1 pairs (56.89%)
  Overlap: 626524 week 2 pairs (58.38%)
  Overlap: 633677 week 3 pairs (59.85%)
  Overlap: 710405 week 4 pairs (64.69%)
  Overlap: 709483 week 5 pairs (62.60%)
  Overlap: 683873 week 6 pairs (58.10%)
  Overlap: 665237 week 7 pairs (56.48%)


In [ ]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.5956933624203062


# 3 Hours

In [9]:
# Time aggregation
agg_hour_level = 3

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [10]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [11]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [12]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [13]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 260254 week 1 pairs (60.89%)
  Overlap: 265449 week 2 pairs (62.82%)
  Overlap: 268752 week 3 pairs (64.51%)
  Overlap: 297580 week 4 pairs (69.01%)
  Overlap: 291873 week 5 pairs (66.13%)
  Overlap: 272054 week 6 pairs (59.89%)
  Overlap: 265632 week 7 pairs (58.26%)


In [14]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.6307293999474097


# 6 Hours

In [15]:
# Time aggregation
agg_hour_level = 6

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [16]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [17]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [18]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [19]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 141712 week 1 pairs (59.66%)
  Overlap: 143750 week 2 pairs (61.34%)
  Overlap: 145464 week 3 pairs (62.85%)
  Overlap: 160811 week 4 pairs (67.16%)
  Overlap: 158552 week 5 pairs (64.74%)
  Overlap: 147642 week 6 pairs (58.69%)
  Overlap: 145471 week 7 pairs (57.39%)


In [20]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.6169114695078098


## 30 Mins

In [22]:
# Time aggregation
agg_hour_level = 0.5

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [23]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [24]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [25]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [26]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 806676 week 1 pairs (44.92%)
  Overlap: 802658 week 2 pairs (45.00%)
  Overlap: 803643 week 3 pairs (45.59%)
  Overlap: 922282 week 4 pairs (50.01%)
  Overlap: 952223 week 5 pairs (49.63%)
  Overlap: 939665 week 6 pairs (47.07%)
  Overlap: 914982 week 7 pairs (45.76%)


In [27]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.46854138854448374


# 10 Mins

In [28]:
# Time aggregation
agg_hour_level = 1/6

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [29]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [30]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [31]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [32]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 1068563 week 1 pairs (29.88%)
  Overlap: 1027703 week 2 pairs (29.04%)
  Overlap: 1021635 week 3 pairs (29.39%)
  Overlap: 1203125 week 4 pairs (32.46%)
  Overlap: 1249242 week 5 pairs (32.32%)
  Overlap: 1213734 week 6 pairs (30.51%)
  Overlap: 1178461 week 7 pairs (29.48%)


In [33]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.30438581177659035


# 1 Minute

In [34]:
# Time aggregation
agg_hour_level = 1/60

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [35]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [36]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [37]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [38]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 1721948 week 1 pairs (15.64%)
  Overlap: 1597250 week 2 pairs (14.54%)
  Overlap: 1749515 week 3 pairs (16.14%)
  Overlap: 2107065 week 4 pairs (17.96%)
  Overlap: 2042019 week 5 pairs (16.97%)
  Overlap: 1991305 week 6 pairs (16.49%)
  Overlap: 1740449 week 7 pairs (14.52%)


In [39]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.16035890652263265


# 4 Hours

In [40]:
# Time aggregation
agg_hour_level = 4

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [41]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [42]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [43]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [44]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 204126 week 1 pairs (60.54%)
  Overlap: 207090 week 2 pairs (62.23%)
  Overlap: 209509 week 3 pairs (63.79%)
  Overlap: 232067 week 4 pairs (68.24%)
  Overlap: 228391 week 5 pairs (65.63%)
  Overlap: 212606 week 6 pairs (59.38%)
  Overlap: 208214 week 7 pairs (57.83%)


In [45]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.6252013324765374


# 8 Hours

In [46]:
# Time aggregation
agg_hour_level = 8

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [47]:
# Build the features dataframe
features_df = build_features(df, agg_hour_level)

In [48]:
# Standardise features
X = features_df.select(feature_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [49]:
# Cluster for each week 
n_weeks = 8
k = 2
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled_week = cluster_preprocess(features_df,X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (features_week.with_columns(pl.Series('cluster',labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

In [50]:
# Percentage overlap on consectuive weeks
overlap_percentages = {}

for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)

    overlap_percentages[week] = len(overlap)/union
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 117531 week 1 pairs (59.93%)
  Overlap: 117281 week 2 pairs (60.83%)
  Overlap: 118394 week 3 pairs (62.02%)
  Overlap: 131472 week 4 pairs (66.56%)
  Overlap: 130545 week 5 pairs (64.64%)
  Overlap: 120784 week 6 pairs (58.25%)
  Overlap: 119382 week 7 pairs (56.98%)


In [51]:
# Average overlap of users across weeks
print(np.array(list(overlap_percentages.values())).mean())

0.6131487434372113
